# Persistent Homology of Knight's Tour Graphs as Metric Spaces

This notebook is analogous to the Möbius ladder notebook, but for **knight's tour graphs** / **knight graphs**.

For a board with `rows = m` and `cols = n`, the knight graph has:

- one vertex for each square `(row, col)`, and
- an edge between two vertices precisely when a chess knight can move between the corresponding squares.

We view the graph as a finite metric space using the **unweighted shortest-path metric** on the graph. Then we compute Vietoris--Rips persistent homology using `ripser(..., distance_matrix=True)`.

Because some small knight graphs are disconnected, this notebook defaults to computing persistence on the **largest connected component**. For sufficiently large square boards this usually coincides with the whole graph, but keeping the component handling explicit makes the experiments safer.

Main tasks:

1. Build square or rectangular knight graphs.
2. Compute graph shortest-path metrics.
3. Compute and plot persistence diagrams and barcodes.
4. Run batch experiments through dimension 3.
5. Run a large-scale positive-dimensional barcode export, producing a text file suitable for later pattern analysis.

In [ ]:
# If needed, uncomment and run this cell once.
# %pip install ripser persim numpy scipy pandas matplotlib ipywidgets networkx

In [ ]:
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, connected_components

from ripser import ripser
from persim import plot_diagrams

try:
    import networkx as nx
    HAS_NETWORKX = True
except Exception:
    HAS_NETWORKX = False

plt.rcParams["figure.figsize"] = (7, 5)

## 1. Construct knight graphs and graph metrics

A standard knight move changes coordinates by `(±1, ±2)` or `(±2, ±1)`. The graph is undirected.

In [ ]:
def square_to_index(r: int, c: int, cols: int) -> int:
    """Convert board coordinates to a vertex index."""
    return r * cols + c


def index_to_square(idx: int, cols: int):
    """Convert a vertex index to board coordinates."""
    return divmod(idx, cols)


def knight_graph_edges(rows: int, cols: int, one: int = 1, two: int = 2):
    """Return undirected edges of the rows x cols knight graph.

    Parameters
    ----------
    rows, cols : int
        Board dimensions.
    one, two : int
        Move lengths. The usual knight graph uses one=1, two=2.

    Returns
    -------
    edges : list[tuple[int, int]]
        Unique undirected edges using linear vertex indices.
    """
    if rows < 1 or cols < 1:
        raise ValueError("rows and cols must both be positive")
    if one <= 0 or two <= 0 or one == two:
        raise ValueError("one and two must be distinct positive integers")

    moves = [
        ( one,  two), ( one, -two), (-one,  two), (-one, -two),
        ( two,  one), ( two, -one), (-two,  one), (-two, -one),
    ]

    edges = set()
    for r in range(rows):
        for c in range(cols):
            u = square_to_index(r, c, cols)
            for dr, dc in moves:
                rr, cc = r + dr, c + dc
                if 0 <= rr < rows and 0 <= cc < cols:
                    v = square_to_index(rr, cc, cols)
                    if u != v:
                        edges.add(tuple(sorted((u, v))))
    return sorted(edges)


def knight_graph_adjacency(rows: int, cols: int, one: int = 1, two: int = 2):
    """Sparse adjacency matrix for the rows x cols knight graph."""
    n_vertices = rows * cols
    edges = knight_graph_edges(rows, cols, one=one, two=two)
    if not edges:
        return csr_matrix((n_vertices, n_vertices), dtype=float)

    row_idx, col_idx = [], []
    for u, v in edges:
        row_idx.extend([u, v])
        col_idx.extend([v, u])
    data = np.ones(len(row_idx), dtype=float)
    return csr_matrix((data, (row_idx, col_idx)), shape=(n_vertices, n_vertices))


def knight_components(rows: int, cols: int, one: int = 1, two: int = 2):
    """Return component labels and sizes for a knight graph."""
    A = knight_graph_adjacency(rows, cols, one=one, two=two)
    n_components, labels = connected_components(A, directed=False)
    sizes = np.bincount(labels, minlength=n_components)
    return labels, sizes


def knight_metric(
    rows: int,
    cols: int,
    one: int = 1,
    two: int = 2,
    component_mode: str = "largest",
):
    """Return a finite shortest-path distance matrix for a knight graph.

    component_mode options:
    - "largest": restrict to the largest connected component.
    - "strict": require the whole graph to be connected; raise an error otherwise.

    Returns
    -------
    D : ndarray
        Finite all-pairs shortest-path distance matrix.
    kept_vertices : ndarray
        Original vertex indices kept in the metric space.
    meta : dict
        Metadata about components and board size.
    """
    A = knight_graph_adjacency(rows, cols, one=one, two=two)
    labels, sizes = knight_components(rows, cols, one=one, two=two)
    n_components = len(sizes)

    if component_mode == "strict" and n_components != 1:
        raise ValueError(f"Knight graph {rows}x{cols} is disconnected with {n_components} components")
    elif component_mode == "largest":
        largest_label = int(np.argmax(sizes))
        kept_vertices = np.where(labels == largest_label)[0]
    else:
        raise ValueError("component_mode must be 'largest' or 'strict'")

    if len(kept_vertices) < 2:
        raise ValueError("Selected component has fewer than two vertices")

    A_sub = A[kept_vertices, :][:, kept_vertices]
    D = shortest_path(A_sub, directed=False, unweighted=True)
    D = np.asarray(D, dtype=float)

    if not np.all(np.isfinite(D)):
        raise ValueError("Distance matrix contains infinite distances; selected part is disconnected")

    meta = {
        "rows": rows,
        "cols": cols,
        "vertices_total": rows * cols,
        "vertices_kept": int(len(kept_vertices)),
        "n_components": int(n_components),
        "component_sizes": sizes.tolist(),
        "edges_total": int(len(knight_graph_edges(rows, cols, one=one, two=two))),
        "component_mode": component_mode,
        "one": one,
        "two": two,
    }
    return D, kept_vertices, meta


def graph_diameter_from_distance_matrix(D):
    """Diameter of a finite metric space represented by a distance matrix."""
    return int(np.nanmax(D[np.isfinite(D)]))


# Quick sanity checks
for side in [3, 4, 5, 6, 8]:
    labels, sizes = knight_components(side, side)
    edges = knight_graph_edges(side, side)
    print(
        f"{side}x{side}: vertices={side*side}, edges={len(edges)}, "
        f"components={len(sizes)}, component sizes={sorted(sizes, reverse=True)[:5]}"
    )

# Metric on largest component for a small example
D5, kept5, meta5 = knight_metric(5, 5, component_mode="largest")
print("5x5 largest-component metric:", D5.shape, "diameter=", graph_diameter_from_distance_matrix(D5), meta5)

## 2. Optional graph visualization

The drawing below uses board coordinates. Persistent homology uses only the path metric, not the drawing geometry.

In [ ]:
def draw_knight_graph(rows: int, cols: int, ax=None, component_mode="all", node_size=80):
    """Draw a knight graph using board coordinates if NetworkX is available."""
    if not HAS_NETWORKX:
        raise ImportError("networkx is not installed. Run `%pip install networkx` or skip this section.")

    edges = knight_graph_edges(rows, cols)
    G = nx.Graph()
    G.add_nodes_from(range(rows * cols))
    G.add_edges_from(edges)

    if component_mode == "largest":
        comps = sorted(nx.connected_components(G), key=len, reverse=True)
        G = G.subgraph(comps[0]).copy()

    pos = {}
    for idx in G.nodes:
        r, c = index_to_square(idx, cols)
        pos[idx] = (c, -r)

    if ax is None:
        fig, ax = plt.subplots(figsize=(max(5, cols * 0.55), max(5, rows * 0.55)))

    nx.draw_networkx_edges(G, pos, ax=ax, width=0.7, alpha=0.6)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_size)
    ax.set_title(f"Knight graph on a {rows}x{cols} board")
    ax.set_aspect("equal")
    ax.axis("off")
    return ax


draw_knight_graph(8, 8, node_size=45);

## 3. Compute Vietoris--Rips persistence with Ripser

The input to Ripser is a shortest-path distance matrix. By default we use the largest connected component, which avoids infinite graph distances for disconnected knight graphs.

In [ ]:
def compute_knight_persistence(
    rows: int,
    cols: int = None,
    maxdim: int = 3,
    coeff: int = 2,
    thresh=None,
    component_mode: str = "largest",
    do_cocycles: bool = False,
):
    """Compute persistent homology of a knight graph with the path-length metric."""
    if cols is None:
        cols = rows

    D, kept_vertices, meta = knight_metric(rows, cols, component_mode=component_mode)
    if thresh is None:
        thresh = graph_diameter_from_distance_matrix(D)

    t0 = time.perf_counter()
    result = ripser(
        D,
        distance_matrix=True,
        maxdim=maxdim,
        coeff=coeff,
        thresh=thresh,
        do_cocycles=do_cocycles,
    )
    elapsed = time.perf_counter() - t0

    result.update(meta)
    result["diameter"] = graph_diameter_from_distance_matrix(D)
    result["thresh"] = thresh
    result["maxdim"] = maxdim
    result["coeff"] = coeff
    result["elapsed_seconds"] = elapsed
    result["kept_vertices"] = kept_vertices
    return result


example = compute_knight_persistence(8, 8, maxdim=3, component_mode="largest")
print(f"Computed {example['rows']}x{example['cols']} in {example['elapsed_seconds']:.3f}s")
print("vertices kept:", example["vertices_kept"], "of", example["vertices_total"])
print("diameter:", example["diameter"])
for dim, dgm in enumerate(example["dgms"]):
    print(f"H_{dim}: {len(dgm)} intervals")
    print(dgm[:10])

## 4. Plot persistence diagrams and barcodes

In [ ]:
def knight_label(result):
    return f"{result['rows']}x{result['cols']} knight graph"


def plot_persistence_diagram(result, title=None):
    """Plot persistence diagrams from a Ripser result dictionary."""
    if title is None:
        title = f"Persistence diagram for {knight_label(result)}"
    plt.figure(figsize=(6, 6))
    plot_diagrams(result["dgms"], show=False)
    plt.title(title)
    plt.show()


def plot_barcode(result, dims=None, sort_by="birth", title=None, inf_extension=0.25):
    """Plot a barcode for selected homology dimensions."""
    dgms = result["dgms"]
    if dims is None:
        dims = list(range(len(dgms)))

    finite_deaths = []
    for dgm in dgms:
        if len(dgm):
            finite_deaths.extend(dgm[np.isfinite(dgm[:, 1]), 1].tolist())
    max_finite = max(finite_deaths) if finite_deaths else 1.0
    inf_value = max_finite + inf_extension * max(1.0, max_finite)

    fig, ax = plt.subplots(figsize=(9, max(3, 0.25 * sum(len(dgms[d]) for d in dims))))
    y = 0
    yticks = []
    yticklabels = []

    for dim in dims:
        intervals = np.asarray(dgms[dim], dtype=float)
        if len(intervals) == 0:
            continue

        if sort_by == "birth":
            order = np.lexsort((intervals[:, 1], intervals[:, 0]))
        elif sort_by == "persistence":
            deaths_for_sort = intervals[:, 1].copy()
            deaths_for_sort[~np.isfinite(deaths_for_sort)] = inf_value
            order = np.argsort(-(deaths_for_sort - intervals[:, 0]))
        else:
            order = np.arange(len(intervals))

        for idx in order:
            birth, death = intervals[idx]
            death_display = inf_value if not np.isfinite(death) else death
            ax.hlines(y, birth, death_display, linewidth=2)
            if not np.isfinite(death):
                ax.plot(death_display, y, marker=">", markersize=6)
            yticks.append(y)
            yticklabels.append(f"H{dim}")
            y += 1

        ax.axhline(y - 0.5, linewidth=0.5, alpha=0.4)

    ax.set_xlabel("Filtration value")
    ax.set_ylabel("Intervals")
    if len(yticks) <= 60:
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels)
    else:
        ax.set_yticks([])
    ax.set_title(title or f"Barcode for {knight_label(result)}")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_persistence_diagram(example)
plot_barcode(example, dims=[0, 1, 2, 3], sort_by="persistence")

## 5. Ordinary batch computation for square boards through dimension 3

Adjust `SQUARE_SIZES`, `MAXDIM`, and `COEFF` as desired. Results are cached to disk with `pickle`.

Warning: Vietoris--Rips complexes of knight graphs can grow quickly. Start conservatively and increase board sizes after checking timings.

In [ ]:
SQUARE_SIZES = list(range(5, 13))  # 5x5 through 12x12 by default
MAXDIM = 3
COEFF = 2
THRESH = None
COMPONENT_MODE = "largest"
CACHE_FILE = Path(f"knight_square_persistence_maxdim{MAXDIM}_coeff{COEFF}.pkl")


def batch_compute_knight_squares(sizes, maxdim=3, coeff=2, thresh=None, component_mode="largest", cache_file=None, force=False):
    """Compute persistence for many square knight graphs and optionally cache results."""
    if cache_file is not None:
        cache_file = Path(cache_file)
        if cache_file.exists() and not force:
            with cache_file.open("rb") as f:
                return pickle.load(f)

    results = {}
    for side in sizes:
        print(f"Computing {side}x{side} knight graph ...", end=" ", flush=True)
        try:
            res = compute_knight_persistence(
                side, side, maxdim=maxdim, coeff=coeff, thresh=thresh, component_mode=component_mode
            )
            results[side] = res
            print(f"done in {res['elapsed_seconds']:.3f}s; diameter={res['diameter']}; kept={res['vertices_kept']}/{res['vertices_total']}", flush=True)
        except Exception as e:
            print(f"FAILED: {e}", flush=True)
            results[side] = {"side": side, "rows": side, "cols": side, "error": repr(e)}

    if cache_file is not None:
        with cache_file.open("wb") as f:
            pickle.dump(results, f)
    return results


results = batch_compute_knight_squares(
    SQUARE_SIZES,
    maxdim=MAXDIM,
    coeff=COEFF,
    thresh=THRESH,
    component_mode=COMPONENT_MODE,
    cache_file=CACHE_FILE,
    force=False,
)

## 6. Summarize persistence across the ordinary batch

In [ ]:
def diagram_stats(dgm):
    """Summary statistics for one persistence diagram."""
    dgm = np.asarray(dgm, dtype=float)
    if len(dgm) == 0:
        return {
            "intervals": 0,
            "finite_intervals": 0,
            "infinite_intervals": 0,
            "max_persistence": 0.0,
            "total_persistence": 0.0,
            "mean_persistence": 0.0,
        }

    finite = np.isfinite(dgm[:, 1])
    pers = dgm[finite, 1] - dgm[finite, 0]
    return {
        "intervals": int(len(dgm)),
        "finite_intervals": int(np.sum(finite)),
        "infinite_intervals": int(np.sum(~finite)),
        "max_persistence": float(np.max(pers)) if len(pers) else 0.0,
        "total_persistence": float(np.sum(pers)) if len(pers) else 0.0,
        "mean_persistence": float(np.mean(pers)) if len(pers) else 0.0,
    }


def summarize_results(results):
    rows = []
    for side, res in results.items():
        if "error" in res:
            rows.append({"side": side, "dimension": None, "error": res["error"]})
            continue
        for dim, dgm in enumerate(res["dgms"]):
            row = {
                "side": side,
                "rows": res["rows"],
                "cols": res["cols"],
                "dimension": dim,
                "vertices_total": res["vertices_total"],
                "vertices_kept": res["vertices_kept"],
                "edges_total": res["edges_total"],
                "n_components": res["n_components"],
                "diameter": res["diameter"],
                "elapsed_seconds": res["elapsed_seconds"],
                "coeff": res["coeff"],
                "maxdim": res["maxdim"],
            }
            row.update(diagram_stats(dgm))
            rows.append(row)
    return pd.DataFrame(rows)


summary = summarize_results(results)
summary.to_csv("knight_square_persistence_summary.csv", index=False)
summary.head(16)

## 7. Plot trends across board size

In [ ]:
def plot_summary_trends(summary, dimensions=None, y="total_persistence"):
    if dimensions is None:
        dimensions = sorted(d for d in summary["dimension"].dropna().unique())

    fig, ax = plt.subplots(figsize=(8, 5))
    for dim in dimensions:
        sub = summary[summary["dimension"] == dim].sort_values("side")
        ax.plot(sub["side"], sub[y], marker="o", label=f"H{int(dim)}")
    ax.set_xlabel("Board side length s for s x s knight graph")
    ax.set_ylabel(y.replace("_", " ").title())
    ax.set_title(f"{y.replace('_', ' ').title()} across square knight graphs")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_summary_trends(summary, dimensions=[0, 1, 2, 3], y="finite_intervals")
plot_summary_trends(summary, dimensions=[0, 1, 2, 3], y="total_persistence")
plot_summary_trends(summary, dimensions=[0, 1, 2, 3], y="max_persistence")

## 8. Inspect one board from the ordinary batch

In [ ]:
SELECTED_SIDE = 8
selected = results[SELECTED_SIDE]

print(f"{SELECTED_SIDE}x{SELECTED_SIDE}: diameter={selected['diameter']}, elapsed={selected['elapsed_seconds']:.3f}s")
print(f"vertices kept={selected['vertices_kept']} of {selected['vertices_total']}; components={selected['n_components']}")
for dim, dgm in enumerate(selected["dgms"]):
    print(f"H_{dim}: {len(dgm)} intervals")

plot_persistence_diagram(selected)
plot_barcode(selected, dims=list(range(MAXDIM + 1)), sort_by="persistence")

## 9. Optional interactive explorer

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    side_widget = widgets.Dropdown(options=sorted(results.keys()), value=sorted(results.keys())[0], description="side")
    dims_widget = widgets.SelectMultiple(
        options=list(range(MAXDIM + 1)),
        value=tuple(range(MAXDIM + 1)),
        description="dims",
    )
    out = widgets.Output()

    def update(change=None):
        with out:
            clear_output(wait=True)
            side = side_widget.value
            dims = list(dims_widget.value)
            res = results[side]
            if "error" in res:
                print(res["error"])
                return
            print(f"{side}x{side}: diameter={res['diameter']}, elapsed={res['elapsed_seconds']:.3f}s")
            print(f"vertices kept={res['vertices_kept']} of {res['vertices_total']}; components={res['n_components']}")
            plot_persistence_diagram(res)
            plot_barcode(res, dims=dims, sort_by="persistence")

    side_widget.observe(update, names="value")
    dims_widget.observe(update, names="value")
    display(widgets.HBox([side_widget, dims_widget]), out)
    update()
except Exception as e:
    print("Interactive widgets are unavailable:", e)

## 10. Export ordinary-batch diagrams as data frames

In [ ]:
def diagrams_to_dataframe(result):
    rows = []
    for dim, dgm in enumerate(result["dgms"]):
        for birth, death in dgm:
            rows.append({
                "rows": result["rows"],
                "cols": result["cols"],
                "side": result["rows"] if result["rows"] == result["cols"] else None,
                "vertices_total": result["vertices_total"],
                "vertices_kept": result["vertices_kept"],
                "dimension": dim,
                "birth": float(birth),
                "death": float(death),
                "persistence": float(death - birth) if np.isfinite(death) else np.inf,
            })
    return pd.DataFrame(rows)


all_intervals = pd.concat(
    [diagrams_to_dataframe(res) for res in results.values() if "error" not in res],
    ignore_index=True,
)
all_intervals.to_csv("knight_square_persistence_intervals.csv", index=False)
all_intervals.head()

## 11. Adaptive large-scale experiment: positive-dimensional bars only

This is the analog of the Möbius ladder pattern-hunting cell.

It computes square knight graphs `side x side`, records all positive-dimensional bars, prints progress, and writes a text file that can be uploaded later for pattern inspection.

This version runs Ripser once per board size with a fixed `LARGE_MAXDIM`, because `ripser(..., maxdim=k)` already computes all dimensions up through `k`.

Practical controls:

- `LARGE_MIN_SIDE`, `LARGE_MAX_SIDE`: square-board side lengths to attempt.
- `LARGE_MAXDIM`: largest homology dimension to compute.
- `LARGE_GLOBAL_TIME_LIMIT_SECONDS`: total runtime cap checked between boards.
- `LARGE_PER_BOARD_SOFT_LIMIT_SECONDS`: once a single board takes longer than this, stop the broad run after recording it.
- `COMPONENT_MODE`: usually `"largest"`, since small knight graphs may be disconnected.

In [ ]:
# ==========================================================
# Efficient large-scale experiment for square knight graphs
# ==========================================================

LARGE_OUTPUT_TXT = "knight_large_scale_positive_dim_bars.txt"
LARGE_OUTPUT_CSV = "knight_large_scale_positive_dim_bars.csv"
LARGE_SUMMARY_CSV = "knight_large_scale_run_summary.csv"

LARGE_MIN_SIDE = 3
LARGE_MAX_SIDE = 13
LARGE_MAXDIM = 3
LARGE_COEFF = 2
LARGE_THRESH = None
LARGE_COMPONENT_MODE = "largest"
LARGE_GLOBAL_TIME_LIMIT_SECONDS = 30 * 60
LARGE_PER_BOARD_SOFT_LIMIT_SECONDS = 180

large_start_time = time.perf_counter()
bar_rows = []
run_rows = []


def write_large_experiment_outputs():
    """Write current accumulated results to TXT and CSV checkpoint files."""
    bars_df = pd.DataFrame(bar_rows)
    runs_df = pd.DataFrame(run_rows)

    bars_df.to_csv(LARGE_OUTPUT_CSV, index=False)
    runs_df.to_csv(LARGE_SUMMARY_CSV, index=False)

    with open(LARGE_OUTPUT_TXT, "w", encoding="utf-8") as f:
        f.write("# Positive-dimensional persistence bars for square knight graphs\n")
        f.write("# Convention: s x s board; path-length metric on selected component; Vietoris--Rips persistence via ripser\n")
        f.write("# Columns: side, rows, cols, dimension, bar_index, birth, death, persistence, maxdim_computed, diameter, vertices_total, vertices_kept, n_components, coeff, thresh\n")
        for row in bar_rows:
            f.write(
                "side={side}, rows={rows}, cols={cols}, dim={dimension}, bar_index={bar_index}, "
                "birth={birth}, death={death}, persistence={persistence}, maxdim_computed={maxdim_computed}, "
                "diameter={diameter}, vertices_total={vertices_total}, vertices_kept={vertices_kept}, "
                "n_components={n_components}, coeff={coeff}, thresh={thresh}\n".format(**row)
            )


print("Starting square-knight large-scale experiment", flush=True)
print("Parameters:", flush=True)
print("  LARGE_MIN_SIDE =", LARGE_MIN_SIDE, flush=True)
print("  LARGE_MAX_SIDE =", LARGE_MAX_SIDE, flush=True)
print("  LARGE_MAXDIM =", LARGE_MAXDIM, flush=True)
print("  LARGE_GLOBAL_TIME_LIMIT_SECONDS =", LARGE_GLOBAL_TIME_LIMIT_SECONDS, flush=True)
print("  LARGE_PER_BOARD_SOFT_LIMIT_SECONDS =", LARGE_PER_BOARD_SOFT_LIMIT_SECONDS, flush=True)

try:
    for side in range(LARGE_MIN_SIDE, LARGE_MAX_SIDE + 1):
        elapsed_global = time.perf_counter() - large_start_time
        if elapsed_global >= LARGE_GLOBAL_TIME_LIMIT_SECONDS:
            print("Global time limit reached before starting next board.", flush=True)
            break

        print("", flush=True)
        print("=" * 72, flush=True)
        print("Starting {}x{} knight graph at global elapsed {:.1f}s".format(side, side, elapsed_global), flush=True)
        print("=" * 72, flush=True)

        try:
            D, kept_vertices, meta = knight_metric(side, side, component_mode=LARGE_COMPONENT_MODE)
            diameter = graph_diameter_from_distance_matrix(D)
            thresh = diameter if LARGE_THRESH is None else LARGE_THRESH

            print(
                "Graph built: vertices_total={}, vertices_kept={}, edges={}, components={}, diameter={}, thresh={}".format(
                    meta["vertices_total"], meta["vertices_kept"], meta["edges_total"], meta["n_components"], diameter, thresh
                ),
                flush=True,
            )
        except Exception as e:
            print("Failed to build {}x{}: {}".format(side, side, repr(e)), flush=True)
            run_rows.append({
                "side": side,
                "rows": side,
                "cols": side,
                "status": "graph_build_failed",
                "maxdim_computed": None,
                "seconds": 0.0,
                "positive_dim_bars": 0,
                "message": repr(e),
            })
            write_large_experiment_outputs()
            continue

        print("Running ripser for {}x{}, maxdim={} ...".format(side, side, LARGE_MAXDIM), end=" ", flush=True)
        t0 = time.perf_counter()

        try:
            result = ripser(
                D,
                distance_matrix=True,
                maxdim=LARGE_MAXDIM,
                coeff=LARGE_COEFF,
                thresh=thresh,
            )
            seconds = time.perf_counter() - t0
            interval_counts = [len(dgm) for dgm in result["dgms"]]
            print("done in {:.2f}s; interval counts={}".format(seconds, interval_counts), flush=True)

            positive_bar_count = 0
            for dim in range(1, len(result["dgms"])):
                dgm = result["dgms"][dim]
                for bar_index, pair in enumerate(dgm):
                    birth = float(pair[0])
                    death = float(pair[1])
                    persistence = float(death - birth) if np.isfinite(death) else np.inf
                    bar_rows.append({
                        "side": side,
                        "rows": side,
                        "cols": side,
                        "dimension": dim,
                        "bar_index": bar_index,
                        "birth": birth,
                        "death": death,
                        "persistence": persistence,
                        "maxdim_computed": LARGE_MAXDIM,
                        "diameter": diameter,
                        "vertices_total": meta["vertices_total"],
                        "vertices_kept": meta["vertices_kept"],
                        "n_components": meta["n_components"],
                        "coeff": LARGE_COEFF,
                        "thresh": thresh,
                    })
                    positive_bar_count += 1

            run_rows.append({
                "side": side,
                "rows": side,
                "cols": side,
                "status": "success",
                "maxdim_computed": LARGE_MAXDIM,
                "seconds": seconds,
                "positive_dim_bars": positive_bar_count,
                "message": "",
            })

            print("Recorded {} positive-dimensional bars for {}x{}.".format(positive_bar_count, side, side), flush=True)
            write_large_experiment_outputs()

            if seconds >= LARGE_PER_BOARD_SOFT_LIMIT_SECONDS:
                print("This board exceeded the per-board soft limit. Stopping broad run.", flush=True)
                break

        except Exception as e:
            seconds = time.perf_counter() - t0
            print("failed after {:.2f}s: {}".format(seconds, repr(e)), flush=True)
            run_rows.append({
                "side": side,
                "rows": side,
                "cols": side,
                "status": "failed",
                "maxdim_computed": LARGE_MAXDIM,
                "seconds": seconds,
                "positive_dim_bars": 0,
                "message": repr(e),
            })
            write_large_experiment_outputs()
            break

except KeyboardInterrupt:
    print("", flush=True)
    print("KeyboardInterrupt received. Writing partial results.", flush=True)
    write_large_experiment_outputs()

print("", flush=True)
print("Square-knight large-scale experiment finished or interrupted safely.", flush=True)
print("Total positive-dimensional bars recorded:", len(bar_rows), flush=True)
print("Text output:", LARGE_OUTPUT_TXT, flush=True)
print("CSV output:", LARGE_OUTPUT_CSV, flush=True)
print("Run summary:", LARGE_SUMMARY_CSV, flush=True)

pd.DataFrame(run_rows).tail()

## 12. Rectangular-board batch template

If you want rectangular knight graphs rather than square boards, edit `RECTANGLE_BOARDS` below. Each pair is `(rows, cols)`.

In [ ]:
RECTANGLE_BOARDS = [(5, 6), (5, 7), (6, 8), (8, 10)]
RECT_MAXDIM = 3

rect_results = {}
for rows, cols in RECTANGLE_BOARDS:
    print(f"Computing {rows}x{cols} ...", end=" ", flush=True)
    try:
        res = compute_knight_persistence(rows, cols, maxdim=RECT_MAXDIM, component_mode="largest")
        rect_results[(rows, cols)] = res
        print(f"done in {res['elapsed_seconds']:.3f}s; kept={res['vertices_kept']}/{res['vertices_total']}; diameter={res['diameter']}", flush=True)
    except Exception as e:
        rect_results[(rows, cols)] = {"rows": rows, "cols": cols, "error": repr(e)}
        print(f"FAILED: {e}", flush=True)

## 13. Ideas for further experiments

- Compare square boards `s x s` to rectangular boards `m x n`.
- Compare largest-component persistence with strict connected-board persistence.
- Record component-size data for small boards; disconnectedness can strongly affect interpretation.
- Try toroidal knight graphs by modifying the edge generator to wrap around board boundaries.
- Compare coefficient fields, e.g. `coeff=2`, `coeff=3`, and `coeff=5`.
- If high-dimensional computations become slow, reduce `LARGE_MAXDIM`, `LARGE_MAX_SIDE`, or the threshold.